---
title: Laboratory Task 
---

# E1 - CNN Implementation

**Name:** Rosemarie Ann S. Bajao  
**Course:** BS Data Science  
**Activity:** PyTorch Basics  

---

## Overview

A Convolutional Neural Network is commonly used for image-related tasks because it can learn useful features from image data. In the original notebook, the MNIST dataset is used, where each image is a grayscale image with a size of 28 × 28 pixels.

The architecture used in this laboratory activity starts with convolution and pooling layers that extract features from the image. The resulting feature maps are then passed through dropout and fully connected layers before producing the final output for the 10 MNIST digit classes.

## Convolutional Neural Network

A CNN processes an image through several layers. Convolutional layers learn image features, while pooling layers reduce the spatial size of the feature maps. Activation functions such as ReLU are applied after the convolutional and fully connected layers in the given architecture.

For the MNIST input, the first layer receives one channel with a height and width of 28 pixels:

$$
	ext{Input} = (1, 28, 28)
$$

## Architecture Diagram

The architecture uses four convolutional layers and two max-pooling layers. The first pooling layer uses padding of 1, so the 28 × 28 input becomes 15 × 15 after pooling. The second pooling layer reduces the 15 × 15 feature maps to 7 × 7.

| Layer | Configuration | Output Shape |
|---|---|---|
| Input | 1 channel, 28 × 28 | `(1, 28, 28)` |
| Conv1 + ReLU | 1 → 32, kernel 3 × 3, padding 1 | `(32, 28, 28)` |
| MaxPool1 | kernel 2 × 2, stride 2, padding 1 | `(32, 15, 15)` |
| Conv2 + ReLU | 32 → 64, kernel 3 × 3, padding 1 | `(64, 15, 15)` |
| Conv3 + ReLU | 64 → 128, kernel 3 × 3, padding 1 | `(128, 15, 15)` |
| Conv4 + ReLU | 128 → 256, kernel 3 × 3, padding 1 | `(256, 15, 15)` |
| MaxPool2 | kernel 2 × 2, stride 2 | `(256, 7, 7)` |
| Dropout | `p = 0.2` | `(256, 7, 7)` |
| Flatten | `256 × 7 × 7` | `12544` |
| FCN1 + ReLU | 12544 → 1000 | `1000` |
| FCN2 + ReLU | 1000 → 500 | `500` |
| FCN3 + SoftMax | 500 → 10 | `10` |

## PyTorch Layers

The convolutional layers can be created using <tt>nn.Conv2d</tt>, while the pooling layers use <tt>nn.MaxPool2d</tt>. The fully connected layers use <tt>nn.Linear</tt>. A dropout layer with <tt>p=0.2</tt> is placed before flattening the feature maps.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

## Laboratory Activity 6

Instruction: Convert the given CNN architecture diagram into a PyTorch CNN Architecture.

The diagram contains four convolutional layers, two max-pooling layers, a dropout layer, three fully connected layers, ReLU activations, and a final SoftMax output.

![CNN architecture diagram](attachment:task6_architecture.png)



In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()

        # Convolutional layers
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2, padding=1)

        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1)
        self.conv4 = nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=1)

        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2, padding=0)

        # Dropout
        self.dropout = nn.Dropout(p=0.2)

        # Fully connected layers
        self.fcn1 = nn.Linear(256 * 7 * 7, 1000)
        self.fcn2 = nn.Linear(1000, 500)
        self.fcn3 = nn.Linear(500, 10)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool1(x)

        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))
        x = F.relu(self.conv4(x))
        x = self.pool2(x)

        x = self.dropout(x)

        x = x.view(x.size(0), -1)

        x = F.relu(self.fcn1(x))
        x = F.relu(self.fcn2(x))
        x = F.softmax(self.fcn3(x), dim=1)

        return x


### Instantiate the CNN

The class can now be instantiated to create the CNN model.

In [ ]:
model = CNN()
model

### Check the Output Shape

Since the MNIST input has one channel and a size of 28 × 28, a sample input can be created with the shape <tt>(32, 1, 28, 28)</tt>, where 32 represents the batch size. The final layer should produce 10 outputs, one for each digit from 0 to 9.

In [ ]:
sample_input = torch.randn(32, 1, 28, 28)

output = model(sample_input)

print("Input shape:", sample_input.shape)
print("Output shape:", output.shape)

### heck the Intermediate Shapes

The intermediate shapes can also be checked to make sure that the architecture follows the given diagram. The first pooling layer changes the spatial size from 28 × 28 to 15 × 15 because it uses padding of 1. The second pooling layer then changes 15 × 15 to 7 × 7.

In [ ]:
x = sample_input

x = F.relu(model.conv1(x))
print("After Conv1:", x.shape)

x = model.pool1(x)
print("After MaxPool1:", x.shape)

x = F.relu(model.conv2(x))
print("After Conv2:", x.shape)

x = F.relu(model.conv3(x))
print("After Conv3:", x.shape)

x = F.relu(model.conv4(x))
print("After Conv4:", x.shape)

x = model.pool2(x)
print("After MaxPool2:", x.shape)

x = model.dropout(x)
x = x.view(x.size(0), -1)
print("After Flatten:", x.shape)

The CNN was created based on the architecture shown in the diagram. The input is a grayscale MNIST image with one channel and a size of 28 × 28. The first convolutional layer changes the number of channels from 1 to 32 while keeping the spatial size at 28 × 28 because a kernel size of 3 and padding of 1 are used.

The first max-pooling layer reduces the spatial dimensions to 15 × 15. Three more convolutional layers then increase the number of feature channels from 32 to 64, then 128, and finally 256. The second max-pooling layer reduces the feature maps to 7 × 7.

After pooling, dropout with a probability of 0.2 is applied. The feature maps are then flattened into 256 × 7 × 7 = 12,544 values before entering the fully connected layers. The first fully connected layer produces 1,000 values, the second produces 500 values, and the final layer produces 10 outputs. ReLU is used in the hidden layers, while SoftMax is applied to the final output to produce class probabilities for the ten MNIST digits.

*Since SoftMax is already applied inside the model, this architecture is meant to be used with a loss function that expects probabilities, such as <tt>nn.NLLLoss</tt> (combined with a log of the output) rather than <tt>nn.CrossEntropyLoss</tt>. <tt>nn.CrossEntropyLoss</tt> already applies SoftMax internally, so combining it with a model that also applies SoftMax would apply it twice, which can affect training. This becomes relevant once this architecture is actually trained in a later activity.*